# TopoConscious Demo Pipeline
End-to-end run on synthetic data.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from topoconscious.topology import PersistenceEngine
from topoconscious.hmm import TopologicalHMM
from topoconscious.transfer_entropy import TopologicalTransferEntropy
from topoconscious.metrics import MuellerLyerCurrent


## 1. Synthetic fMRI time-series (300 volumes, 90 regions)

In [ ]:
rng = np.random.default_rng(42)
ts = rng.standard_normal((300, 90))
for i in range(45):
    ts[:150, i] += 0.5 * ts[:150, 0]
print('Time series shape:', ts.shape)

## 2. Sliding-window persistent homology

In [ ]:
engine = PersistenceEngine(max_dim=2, n_landmarks=50)
window_size, step = 30, 5
windows = [ts[t:t+window_size] for t in range(0, 300-window_size, step)]
print(f'Number of windows: {len(windows)}')
diagrams = [engine.compute(w) for w in windows]
print('Computed diagrams for all windows.')

## 3. Wasserstein + Müller-Lyer timelines

In [ ]:
wt = engine.wasserstein_timeline(diagrams)
ml = MuellerLyerCurrent(alpha=0.1, beta=0.05)
ml_tl = ml.timeline(diagrams, dim=1)
from topoconscious.metrics import PersistenceLandscape
pl = PersistenceLandscape(n_landscapes=5, resolution=100)
pl_tl = pl.timeline(diagrams, dim=1)

fig, axes = plt.subplots(3, 1, figsize=(12, 7), sharex=True)
axes[0].plot(wt, color='tomato', label='Wasserstein W₂'); axes[0].set_ylabel('W₂'); axes[0].legend()
axes[1].plot(ml_tl, color='purple', label='Müller-Lyer current'); axes[1].set_ylabel('ML dist'); axes[1].legend()
axes[2].plot(pl_tl, color='teal', label='Persistence Landscape L2'); axes[2].set_ylabel('PL dist'); axes[2].set_xlabel('Window'); axes[2].legend()
plt.suptitle('Topological Distance Timelines (H₁) — Three Metrics')
plt.tight_layout()
plt.show()

## 4. HMM consciousness decoding

In [ ]:
sig = engine.signature_vectors(diagrams)
model = TopologicalHMM(n_states=2)
result = model.fit_decode(sig)
plt.figure(figsize=(12,3))
plt.fill_between(range(len(result['p_conscious'])), result['p_conscious'], alpha=0.7, color='steelblue')
plt.axhline(0.5, color='red', linestyle='--')
plt.title('P(conscious) – HMM Output')
plt.ylim(0,1); plt.xlabel('Window'); plt.tight_layout(); plt.show()

## 5. Topological Transfer Entropy

In [ ]:
calc = TopologicalTransferEntropy(lag=1, n_bins=8)
te = calc.compute(diagrams, ts)
plt.figure(figsize=(7,6))
plt.imshow(te, cmap='hot', aspect='auto')
plt.colorbar(label='TE (bits)')
plt.title('Topological Transfer Entropy Matrix')
plt.xlabel('Target region'); plt.ylabel('Source region')
plt.tight_layout(); plt.show()